# Module 14: Structured Products & Correlation

This notebook demonstrates the models for Default Correlation (Gaussian & Student-t Copulas) and Tranche Losses (Equity, Mezzanine, Senior) used in Synthetic CDOs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.structured.copula import GaussianCopula, StudentTCopula
from src.structured.tranche import Tranche
from src.structured.cdo import SyntheticCDO

np.random.seed(42)

## 1. Tranche Sensitivities to Correlation (Correlation Smile)

In [ ]:
n_credits = 100
hazard_rates = np.ones(n_credits) * 0.02  # 2% hazard rate
recovery_rate = 0.4
time_horizon = 5.0

tranches = [
    Tranche(0.0, 0.03, "Equity"),
    Tranche(0.03, 0.10, "Mezzanine"),
    Tranche(0.10, 1.0, "Senior"),
]

correlations = np.linspace(0.0, 0.9, 10)
equity_losses = []
mezzanine_losses = []
senior_losses = []

for rho in correlations:
    corr = np.eye(n_credits) * rho + (1 - rho) * np.eye(n_credits)
    corr[corr == 0] = rho

    copula = GaussianCopula(corr)
    cdo = SyntheticCDO(tranches, hazard_rates, recovery_rate)

    results = cdo.simulate_tranche_losses(copula, time_horizon, n_samples=5000, seed=42)

    equity_losses.append(results["Equity"]["expected_loss"])
    mezzanine_losses.append(results["Mezzanine"]["expected_loss"])
    senior_losses.append(results["Senior"]["expected_loss"])

plt.figure(figsize=(10, 6))
plt.plot(correlations, equity_losses, label="Equity (0-3%)", marker="o")
plt.plot(correlations, mezzanine_losses, label="Mezzanine (3-10%)", marker="s")
plt.plot(correlations, senior_losses, label="Senior (10-100%)", marker="^")
plt.xlabel("Base Correlation (rho)")
plt.ylabel("Expected Tranche Loss (%)")
plt.title("Tranche Losses vs Correlation (Gaussian Copula)")
plt.legend()
plt.grid(True)
plt.show()